In [1]:
import pandas as pd
import numpy as np
import re
import networkx as nx
from matplotlib.lines import Line2D
import os
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm
import matplotlib.pyplot as plt
import pylab as plt
import openpyxl

In [2]:
BTLA_FILE      = "BTLAvsTCR_24h.csv"
COMPOUND_FILE  = "cd8_limma_merged_filtered_targets_ic50.csv"

cd8_btla = pd.read_csv(BTLA_FILE)
cd8_btla["gene"] = cd8_btla["gene"].astype(str).str.strip()   # in case of trailing whitespace
cd8_btla = cd8_btla.set_index("gene")

btla_lookup = cd8_btla["stat"]
print(f"BTLA reference genes: {len(cd8_btla):,}")

df = pd.read_csv(COMPOUND_FILE)\
    .rename(columns={"Unnamed: 0": "sample", "Unnamed: 1": "row_idx"})

df_avg = (
    df.groupby(["compound_name", "gene"])
    .agg(logFC=("logFC", "mean"), adj_P_Val=("adj.P.Val", "mean"))
    .reset_index()
)
df_avg["logFC_thresh"] = np.where(df_avg["adj_P_Val"] < 0.05, df_avg["logFC"], 0)

print(f"Significant genes kept  : {(df_avg['adj_P_Val'] < 0.05).sum():,}")
print(f"Insignificant → zeroed  : {(df_avg['adj_P_Val'] >= 0.05).sum():,}")

pivot = df_avg.pivot_table(
    index   = "compound_name",
    columns = "gene",
    values  = "logFC_thresh",
    aggfunc = "first"
).fillna(0)

BTLA reference genes: 16,725
Significant genes kept  : 60,158
Insignificant → zeroed  : 0


In [3]:
# 4. Build STV_btla

common_genes  = pivot.columns.intersection(btla_lookup.index)
pivot_aligned = pivot[common_genes]
btla_aligned  = btla_lookup[common_genes].values

STV_btla = btla_aligned / np.linalg.norm(btla_aligned)

# 5. DPD per compound = dot(compound logFC vector, STV_btla)

dpd_vec = pivot_aligned.values @ STV_btla

dpd_df = pd.DataFrame({
    "compound_name" : pivot_aligned.index,
    "dpd"           : dpd_vec.round(4),
    "direction"     : ["drives_activation" if x > 0 else "drives_resting"
                       for x in dpd_vec]
}).sort_values("dpd", ascending=False).reset_index(drop=True)

# 6. Add targets and mechanism

targets = pd.read_csv(COMPOUND_FILE)[
    ["compound_name", "target_protein", "mechanism"]
].drop_duplicates(subset="compound_name")

dpd_df = dpd_df.merge(targets, on="compound_name", how="left")

# 7. Save
dpd_df.to_csv("dpd_sum_per_compound_raw_btla.csv", index=False)
print("\n✓ Saved dpd_sum_per_compound_raw_btla.csv")



✓ Saved dpd_sum_per_compound_raw_btla.csv


In [4]:
STV_MDM2 = pivot.loc["Serdemetan"].values
STV_MDM2 = STV_MDM2 / np.linalg.norm(STV_MDM2)


In [5]:
STV_MDM2

array([ 0.        ,  0.        ,  0.        , ...,  0.        ,
       -0.00464565, -0.00833597], shape=(15045,))

In [6]:
top4 = dpd_df.nlargest(4, "dpd")
bottom4 = dpd_df.nsmallest(4, "dpd")

top_bottom4 = pd.concat([top4, bottom4]).reset_index(drop=True)
top_bottom4


,compound_name,dpd,direction,target_protein,mechanism
0,Sapanisertib,5.2980,drives_activation,Serine/threonine-protein kinase mTOR,Inhibitor
1,I-BET 762,2.7414,drives_activation,"BET bromodomain proteins (BRD2, BRD3, BRD4)",Inhibitor
2,Dexamethasone,1.2519,drives_activation,Glucocorticoid receptor (NR3C1),Activator
3,Temsirolimus,0.4833,drives_activation,Serine/threonine-protein kinase mTOR,Inhibitor
4,ruxolitinib,-6.0264,drives_resting,JAK1; JAK2,Inhibitor
5,Serdemetan,-5.5098,drives_resting,HDM2 (MDM2),Inhibitor
6,panobinostat,-1.3395,drives_resting,Histone deacetylase 1 (HDAC1); HDAC2; pan-HDAC...,Inhibitor
7,BMS-536924,-1.0781,drives_resting,Insulin-like growth factor 1 receptor (IGF-1R),Inhibitor


In [7]:
top_bottom4.to_csv("top4_bottom4_dpd_btla.csv", index=False)
